# MMA-DFER + UOT — Kaggle pipelineMục đích: **kiểm chứng pipeline chạy đúng**, không phải lấy số cuối cùng(Kaggle giới hạn ~12h/phiên nên không đủ cho 5 fold × 25 epoch).Chạy tuần tự. Chỉ cần sửa **Cell 2 (CONFIG)** — các cell sau tự dùng lại biến ở đó.Tài liệu: `KAGGLE.md`, `SETUP.md`, `UOT_INTEGRATION.md`, `UOT_CODE_WALKTHROUGH.md`.

## 1. Clone repo

In [ ]:
REPO   = "https://github.com/YouttyLe-DSAI/DEFR-UOT.git"BRANCH = "feat/uot-fusion"WORK   = "/kaggle/working/DEFR-UOT"import os, shutilif os.path.exists(WORK):    shutil.rmtree(WORK)          # rerun-safe!git clone -b {BRANCH} --depth 1 {REPO} {WORK}%cd {WORK}!git log --oneline -1# Repo private? Lưu token trong Add-ons -> Secrets (tên: GH_TOKEN), rồi dùng:# from kaggle_secrets import UserSecretsClient# tok = UserSecretsClient().get_secret("GH_TOKEN")# !git clone -b {BRANCH} https://{tok}@github.com/YouttyLe-DSAI/DEFR-UOT.git {WORK}

## 2. CONFIG — cell duy nhất cần sửaMặc định `FRAMES`/`AUDIO` để trống → **Cell 5 tự dò theo nội dung**, không cần biếtslug Kaggle. Tên hiển thị trong sidebar là *tiêu đề* dataset, còn đường dẫn mount dùng*slug* — hai thứ này thường khác nhau, nên đừng chép tay từ sidebar.Chỉ điền tay khi auto dò sai/thiếu, và lấy đường dẫn thật từ output Cell 4.

In [ ]:
DATASET  = "MAFW"                          # "MAFW" hoặc "DFEW"DATA     = "/kaggle/temp/data"             # cây symlink; /kaggle/temp không tính vào quota 20GBCKPT_DIR = "/kaggle/input/mma-dfer-pretrained"   # chứa 2 file .pth, xem Cell 7# Tham số train — chọn cho GPU Kaggle (T4 16GB, 2-4 vCPU)EPOCHS, BATCH, LR, WORKERS, FOLD = 5, 4, 7e-5, 2, 1# Để trống = tự dò. Chỉ điền khi Cell 5 báo dò thiếu.FRAMES = []AUDIO  = []ANN = (f"annotation/MAFW_set_{FOLD}_train_faces.txt" if DATASET == "MAFW"       else f"annotation/DFEW_set_{FOLD}_train.txt")ANN_TEST    = ANN.replace("train", "test")FRAMES_ROOT = f"{DATA}/mfaw/clips_faces" if DATASET == "MAFW" else f"{DATA}/dfew/clip_224x224"SRC_ARG     = ("--auto" if not FRAMES               else f'--frames {" ".join(FRAMES)} --audio {" ".join(AUDIO)}')print(DATASET, "| nguồn:", "tự dò" if not FRAMES else "chỉ định tay")print("frames root sẽ dựng tại:", FRAMES_ROOT)

## 3. Dependencies`timm==0.9.16` là **bắt buộc**: image Kaggle dùng timm 1.x, mà `models/models_vit.py`kế thừa `timm.models.vision_transformer.VisionTransformer` — API đổi giữa 2 major version.Không cài đè torch/torchaudio (sẽ mất bản CUDA của Kaggle).

In [ ]:
!pip install -q timm==0.9.16 einops==0.7.0 librosa==0.10.1import torch, timmprint("torch", torch.__version__, "| timm", timm.__version__,      "| GPU", torch.cuda.device_count(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 4. Xem chính xác cái gì đang được mountIn cây 3 tầng của mọi dataset, tự đánh dấu chỗ nào là *clip folders*, chỗ nào là *audio*.Cell này chỉ để **đối chiếu bằng mắt**. Nếu Cell 5 tự dò ra đúng thì không cần chép gì cả.

In [ ]:
!python tools/kaggle_setup.py --diagnose

## 5. Dựng cây symlinkDataloader suy ra đường dẫn `.wav` **từ đường dẫn frame bằng phép thay chuỗi**(`clips_faces`→`clips_wav`, `clip_224x224`→`raw_wav`) và chọn nhánh bằng substring(`mfaw`, `clip_224x224`). Trên Kaggle frames nằm ở nhiều mount khác nhau và audio ởmount thứ ba, nên phép thay chuỗi đó **không thể hoạt động**.Cell này dựng đúng cấu trúc mà loader mong đợi bằng symlink (tốn ~0 dung lượng),**không sửa một dòng nào của code baseline**.Với `--auto`, script dò theo **nội dung** chứ không theo tên: thư mục nào có ≥5 thư mụccon mà bên trong là ảnh → *frames root*; thư mục nào chứa `.wav` → *audio root*; rồi lọctheo `mafw`/`mfaw` hoặc `dfew` để không lẫn hai dataset với nhau.Đọc kỹ output:- `LAYOUT OK` — bắt buộc- số `FRAMES` dò được phải khớp số shard bạn attach (MAFW: 2, DFEW: 4)- `clips w/o wav` — **phải = 0 với DFEW** (không có fallback). Với MAFW phải nhỏ:  loader thay wav thiếu bằng `torch.zeros(512,128)`, nhiều quá thì audio chết âm thầm  và toàn bộ so sánh UOT trở nên vô nghĩa.

In [ ]:
!python tools/kaggle_setup.py --dataset {DATASET} {SRC_ARG} --out {DATA}

## 6. Trỏ annotation vào cây vừa dựng`--recount` bắt buộc: bộ preprocess của bạn gần như chắc chắn ra số frame khác bản gốc,mà loader dùng cột đó để sample index.Điều kiện đi tiếp: `missing folder: 0`, `frame count off: 0`, **`UNMATCHED path: 0`**,`labels seen` = `0..10` (MAFW) hoặc `0..6` (DFEW).

In [ ]:
!cp -r annotation annotation.bak!python tools/retarget_annotations.py --dataset {DATASET} --new-root {FRAMES_ROOT} --recount --drop-missingprint("=" * 60)!python tools/check_data.py --annotation {ANN} --n 300print("=" * 60)!python tools/check_data.py --annotation {ANN_TEST} --n 300

## 7. Checkpoint pretrainCần 2 file (tên bị hardcode trong `models/Generate_Model.py`, phải đúng chính xác):| File | ~Size | Nguồn ||---|---|---|| `mae_face_pretrain_vit_base.pth` | 1.3 GB | https://github.com/FuxiVirtualHuman/MAE-Face/releases || `audiomae_pretrained.pth` | 1.2 GB | https://github.com/facebookresearch/AudioMAE |Tải về máy → tạo Kaggle Dataset → Add Input → sửa `CKPT_DIR` ở Cell 2.

In [ ]:
!cp {CKPT_DIR}/mae_face_pretrain_vit_base.pth .!cp {CKPT_DIR}/audiomae_pretrained.pth .!ls -la *.pth

## 8. Smoke test — đừng bỏ quaTốn ~1 phút, bắt đúng các lỗi mà nếu không sẽ chỉ lộ ra sau vài giờ train.Cần thấy:- `SMOKE TEST PASSED`- gradient của **gate** khác 0 (gradient của `proj`/`norm` **bằng 0 là đúng** ở step đầu:  chúng nằm sau `tanh(gate)=0`)- `max|baseline - uot| ... OK` — xác nhận model khởi đầu trùng khít baseline- `peak GPU memory` — cho biết `BATCH` nào vừa VRAM

In [ ]:
!python tools/smoke_test.py --dataset {DATASET} --use-uot --batch-size 2

## 9. Train — baselineChạy baseline **trước** để có mốc so sánh. Cùng seed (hardcode `seed=1`), cùng fold,cùng số epoch với run UOT.`--folds` là cờ thêm vào `main.py`: mặc định script gốc chạy cả 5 fold liên tiếp trongmột process, không thể xong trong giới hạn phiên của Kaggle.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --exper-name KAGGLE_BASE

## 10. Train — UOTKhác đúng một thứ: `--use-uot`. Ablation quan trọng nhất là `--uot-tau 1e6`(≈ balanced OT) — nếu nó ngang bằng `--uot-tau 1.0` thì tính "unbalanced" không đónggóp gì. Chi tiết các nút vặn: `UOT_CODE_WALKTHROUGH.md` §3.

In [ ]:
!python main.py --dataset {DATASET} --folds {FOLD} --epochs {EPOCHS} \  --batch-size {BATCH} --workers {WORKERS} --lr {LR} --weight-decay 1e-2 \  --print-freq 20 --temporal-layers 1 --img-size 224 \  --use-uot --uot-eps 0.05 --uot-tau 1.0 --uot-iters 10 \  --exper-name KAGGLE_UOT_tau1.0

## 11. Kết quả

In [ ]:
import glob, refor log in sorted(glob.glob("log/*/log.txt")):    txt = open(log).read()    accs = re.findall(r"Current Accuracy: ([\d.]+)", txt)    uar  = re.findall(r"UAR: ([\d.]+)", txt)    war  = re.findall(r"WAR: ([\d.]+)", txt)    ep   = re.findall(r"An epoch time: ([\d.]+)", txt)    name = log.split("/")[1]    print(f"{name}")    print(f"   val acc mỗi epoch : {accs}")    print(f"   UAR / WAR         : {uar} / {war}")    if ep:        m = sum(float(e) for e in ep) / len(ep) / 60        print(f"   epoch trung bình  : {m:.1f} phút  ->  25 epoch x 5 fold ~ {m*25*5/60:.1f} giờ")    print()

## Ghi chú- Kết quả nằm ở `/kaggle/working/DEFR-UOT/log/` → được giữ khi **Save Version** (quota 20 GB).- Phiên tương tác bị ngắt khi idle. Chạy dài thì dùng **Save Version → Save & Run All (Commit)**.- `/kaggle/temp` bị xoá sau mỗi phiên → chạy lại Cell 5 mỗi lần mở notebook (vài giây).- Dùng số `epoch trung bình` ở Cell 11 để ước lượng ngân sách thật trên server.